# Sampling Combinatorial Variants from Activity Bins

After independently screening each fragment in the library (holding the other
fragments constant to the parent), we know the first-order activity of every
fragment. A model trained on this data would simply recommend combining the
best fragments — we can do that by hand.

Instead, the goal of the next round is to collect **higher-order interaction**
data by testing combinations of fragments drawn from different activity bins.
This gives the surrogate model information about epistatic effects and helps it
learn which fragment combinations synergize.

This notebook demonstrates:
1. Loading processed fragment-level activity data
2. Defining activity bins (thresholds)
3. Sampling combinatorial variants with weighted fragment sampling
4. Allocating samples across plates
5. Exporting FASTA files for ordering

See the [README section on early rounds](../README.md#-early-rounds-of-testing)
for the experimental rationale behind this approach.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

## Configuration

Adjust these parameters for your experiment. The `connector` string joins
fragment names into a full construct name (e.g. `1.g20___2.g20___3.g20`).

In [ ]:
connector = "___"
num_fragments = 5
num_to_sample_per_bin = 200
valid_wells_per_plate = 382  # 384 - 2 controls (positive + negative)

parent_fragment_names = ["1.g20", "2.g20", "3.g20", "4.g20", "5.g20"]
parent_fragment_seqs = {
    "1.g20": "MGEEEELELERPSGERTPVRRHRFPARKANNIEEAVANVERL",
    "2.g20": "IEEIEASGITFTATADRAVVVGWSLGVITGMIMHATGTDFITAL",
    "3.g20": "RKALEIGKKVKEEDPEFMERHKKIVTDGNRAEIREDIDYWIE",
    "4.g20": "VIEKETGHPVDRSRIFIAETVEEAVELARRAVELGHA",
    "5.g20": "IIVLPPYLIGTAGEAVVQVLASAGVDVLLMGGLGSGVPVTIYRA",
}
parent_name = connector.join(parent_fragment_names)
parent_sequence = "".join([parent_fragment_seqs[f] for f in parent_fragment_names])

## 1. Load Data

Load the processed and filtered experimental data from independent fragment
screening. Each row is one construct with a `name` column containing the
fragment combination (e.g. `1.frag_a___2.g20___3.g20___4.g20___5.g20`
for a single-fragment swap) and a `g20_norm_rate` column with the activity
normalized to the g20 parent.

We also need a mapping from fragment name → amino acid sequence. This can
come from a FASTA of the full library or from the fragment dictionary JSON.

In [ ]:
# --- Load processed experimental data ---
# Replace this path with your own processed data CSV
data_path = Path("../example_data/data_processing/output/260128_processed_filtered_data_round4.csv")
data = pd.read_csv(data_path)

# Parse fragment names from the construct name
frag_cols = [f"frag{i+1}" for i in range(num_fragments)]
data[frag_cols] = data["name"].str.split(connector, expand=True)

# Clean any trailing suffixes (e.g. ":bin3") from the last fragment
data[frag_cols[-1]] = data[frag_cols[-1]].apply(lambda x: x.split(":")[0])

print(f"Loaded {len(data)} constructs")
data[["name", "g20_norm_rate"] + frag_cols].head()

In [ ]:
# --- Load fragment name → sequence mapping ---
# Option A: from a library FASTA file
fasta_dir = Path("../example_data/data_processing/input/fastas")
frag2seq = dict(parent_fragment_seqs)  # seed with parent fragments

for fasta_file in sorted(fasta_dir.glob("*.fasta")):
    with open(fasta_file) as f:
        lines = f.readlines()
    for i in range(0, len(lines), 2):
        header = lines[i].strip().lstrip(">")
        # Headers may be "well;frag_name" — extract the fragment name part
        name = header.split(";")[-1] if ";" in header else header
        # Strip any trailing tags like ":bin3"
        name = name.split(":")[0]
        seq = lines[i + 1].strip()
        # Only keep fragment-level entries (single fragment, not full constructs)
        if connector not in name and seq:
            frag2seq[name] = seq

print(f"Fragment sequence map contains {len(frag2seq)} entries")

## 2. Helper Functions

Two key utilities:
- **`make_all_combinations`**: Exhaustively enumerate all combinations of
  fragment lists (used for the top-k all-by-all block).
- **`weighted_fragment_sampling`**: Sample random combinations with
  inverse-count weighting so that rare fragments are not under-represented.

In [ ]:
def make_all_combinations(frag_lists, index=0, to_return=None):
    """Exhaustively enumerate all fragment combinations.

    Args:
        frag_lists: List of lists, each inner list contains (name, sequence)
                    tuples for one fragment position.

    Returns:
        List of (combined_name, combined_sequence) tuples.
    """
    if to_return is None:
        to_return = []

    if index == len(frag_lists):
        return [
            (connector.join(names), "".join(frags))
            for names, frags in to_return
        ]

    updated = []
    for name, frag in frag_lists[index]:
        if not to_return:
            updated.append(([name], [frag]))
        else:
            for names, frags in to_return:
                updated.append((names + [name], frags + [frag]))

    return make_all_combinations(frag_lists, index + 1, to_return=updated)


def weighted_fragment_sampling(frag_lists, n, max_iters=10000):
    """Sample n unique fragment combinations with inverse-count weighting.

    Fragments that have been sampled fewer times get higher probability,
    promoting uniform coverage across all available fragments.

    Args:
        frag_lists: List of lists of (name, sequence) tuples per position.
        n: Number of unique combinations to sample.
        max_iters: Safety cap on sampling attempts.

    Returns:
        List of (combined_name, combined_sequence) tuples.
    """
    counts = [np.ones(len(frags)) for frags in frag_lists]
    sampled = []

    for _ in range(max_iters):
        if len(sampled) >= n:
            break

        names, seqs = [], []
        for j, frags in enumerate(frag_lists):
            weights = 1.0 / counts[j]
            weights /= weights.sum()
            idx = np.random.choice(len(frags), p=weights)
            names.append(frags[idx][0])
            seqs.append(frags[idx][1])
            counts[j][idx] += 1

        combo = (connector.join(names), "".join(seqs))
        if combo not in sampled:
            sampled.append(combo)

    return sampled

## 3. Well Map

Generate a 384-well plate map for mapping sampled sequences to plate positions.

In [ ]:
plate_rows = list("ABCDEFGHIJKLMNOP")
plate_cols = list(range(1, 25))
wp384 = [f"{r}{c}" for r in plate_rows for c in plate_cols]
print(f"{len(wp384)} wells per plate")

## 4. Define Activity Bins and Sample

We define bins by setting activity thresholds on `g20_norm_rate`. For each
bin, we collect the fragments that pass the threshold at each position and
sample random combinations with weighted fragment sampling.

The first block is special: we take the **top 2** fragments per position and
enumerate **all** combinations (2^5 = 32 for 5 fragments).

Adjust `activity_thresholds` to match your data's distribution.

In [ ]:
# Visualize the per-fragment activity distribution to inform threshold choices
fig, axes = plt.subplots(1, num_fragments, figsize=(4 * num_fragments, 4), sharey=True)
for i in range(num_fragments):
    frag_data = data[data["sample_number"] == i]
    axes[i].hist(frag_data["g20_norm_rate"], bins=20, edgecolor="black", alpha=0.7)
    axes[i].set_xlabel("g20_norm_rate")
    axes[i].set_title(f"Fragment {i+1}")
axes[0].set_ylabel("Count")
plt.suptitle("Activity Distribution per Fragment Position")
plt.tight_layout()
plt.show()

In [ ]:
activity_thresholds = [1.25, 1.0, 0.75, 0.50, 0.25, 0.0]

def get_fragments_above_threshold(threshold):
    """For each fragment position, get (name, sequence) tuples for fragments
    with activity above the given threshold."""
    frag_lists = []
    for i in range(num_fragments):
        frag_data = data[(data["sample_number"] == i) & (data["g20_norm_rate"] > threshold)]
        frag_data = frag_data.sort_values(by="g20_norm_rate", ascending=False)
        frag_names = frag_data[f"frag{i+1}"].tolist()

        frags = []
        for name in frag_names:
            if name in parent_fragment_seqs:
                frags.append((name, parent_fragment_seqs[name]))
            elif name in frag2seq:
                frags.append((name, frag2seq[name]))
        frag_lists.append(frags)
    return frag_lists


# --- Block 1: Top-2 all-by-all combinations ---
top_frags = []
for i in range(num_fragments):
    frag_data = data[data["sample_number"] == i].sort_values(
        by="g20_norm_rate", ascending=False
    )
    top_names = frag_data[f"frag{i+1}"].head(2).tolist()
    top_frags.append([
        (name, parent_fragment_seqs.get(name, frag2seq.get(name, "")))
        for name in top_names
    ])

top_2_combos = make_all_combinations(top_frags)
top_2_combos = [x for x in top_2_combos if x[0] != parent_name]
print(f"Top-2 all-by-all: {len(top_2_combos)} combinations")


# --- Bin sampling ---
bin_combos = {}
for threshold in activity_thresholds:
    frags = get_fragments_above_threshold(threshold)
    combos = weighted_fragment_sampling(frags, num_to_sample_per_bin)
    combos = [x for x in combos if x[0] != parent_name]
    bin_combos[threshold] = combos
    print(f"Bin > {threshold:.2f}: {len(combos)} sampled combinations")

## 5. Allocate Across Plates

Distribute the sampled combinations across 384-well plates. Each plate
reserves 2 wells for controls (positive parent + negative).

Adjust the bin-to-plate allocation below to match your budget.

In [ ]:
sorted_thresholds = sorted(activity_thresholds, reverse=True)

# Plate 1: top-2 combos + bins from the 3 highest thresholds
plate_1 = [(name + ":top2combos", seq) for name, seq in top_2_combos]

remaining = valid_wells_per_plate - len(plate_1)
per_bin_plate1 = remaining // 3

for j, thresh in enumerate(sorted_thresholds[:3]):
    n_take = per_bin_plate1 + (remaining % 3 if j == 0 else 0)
    for combo in bin_combos[thresh][:n_take]:
        plate_1.append((combo[0] + f":bin_gt{thresh}", combo[1]))

# Plate 2: bins from the 3 lowest thresholds
plate_2 = []
per_bin_plate2 = valid_wells_per_plate // 3

for j, thresh in enumerate(sorted_thresholds[3:]):
    n_take = per_bin_plate2 + (1 if j < valid_wells_per_plate % 3 else 0)
    for combo in bin_combos[thresh][:n_take]:
        plate_2.append((combo[0] + f":bin_gt{thresh}", combo[1]))

# Add controls to each plate
controls = [
    (parent_name + ":positive_control", parent_sequence),
    ("negative_control", ""),
]
plate_1.extend(controls)
plate_2.extend(controls)

print(f"Plate 1: {len(plate_1)} wells")
print(f"Plate 2: {len(plate_2)} wells")

## 6. Export FASTA Files

Write one FASTA per plate. Each header contains the well position and the
construct name separated by a semicolon, e.g. `>A1;1.frag_a___2.g20___...:bin_gt1.0`.

In [ ]:
output_dir = Path("../example_data")

def write_plate_fasta(plate_samples, plate_name, output_path):
    lines = []
    for i, (name, seq) in enumerate(plate_samples):
        well = wp384[i]
        lines.append(f">{well};{name}\n")
        lines.append(f"{seq}\n")

    with open(output_path, "w") as f:
        f.writelines(lines)
    print(f"Wrote {len(plate_samples)} sequences to {output_path}")


# Uncomment the lines below to actually write the FASTA files:
# write_plate_fasta(plate_1, "plate_1", output_dir / "bin_sampling_plate_1.fasta")
# write_plate_fasta(plate_2, "plate_2", output_dir / "bin_sampling_plate_2.fasta")

print("\nPreview — first 5 entries from Plate 1:")
for name, seq in plate_1[:5]:
    print(f"  {name[:60]}...  ({len(seq)} aa)")

## Summary

| Block | Source | Count |
|---|---|---|
| Top-2 all-by-all | Best 2 fragments per position, exhaustive | ~32 |
| High activity bins | Fragments above decreasing thresholds | ~200 each |
| Controls | Positive (parent) + negative | 2 per plate |

This sampling strategy provides:
- **Exploitation** via the top-2 combinations
- **Exploration of epistasis** via combinations from different activity tiers
- **Balanced fragment coverage** via weighted sampling

After testing these plates, the data can be combined with earlier rounds and
used to train a surrogate model. See
[`model_data_preparation.ipynb`](model_data_preparation.ipynb) for
formatting the training data.